# AI-Lake × Airbyte Destination Connector

This notebook demonstrates `airbyte-destination-ailake` — the Airbyte CDK v3 destination
connector that writes Airbyte stream records into AI-Lake vector tables.

No Airbyte platform or Docker required here — we use the connector classes directly.

| Section | What you will learn |
|---|---|
| 1 | **Architecture** — how the connector maps streams → tables |
| 2 | **Config** — `AilakeDestinationConfig`, all options |
| 3 | **CmdEmbedder** — local embedding via shell command |
| 4 | **StreamWriter** — batch write + intermediate flush |
| 5 | **Simulate `Destination.write()`** — full connector flow with state messages |
| 6 | **Search results** — AI-Lake vector search on written data |
| 7 | **DuckDB / Iceberg compat** — data readable without the AI-Lake SDK |
| 8 | **Embedding model tracking** — `ailake.embedding-model` in Iceberg properties |
| 9 | **Airbyte platform** — self-hosted UI and PyAirbyte usage |

## 0. Setup

In [ ]:
import json
import math
import os
import pathlib
import random
import sys
import tempfile

import duckdb
import numpy as np

# connector package (installed in the demo container)
from airbyte_destination_ailake.config import AilakeDestinationConfig
from airbyte_destination_ailake.embedder import CmdEmbedder, build_embedder
from airbyte_destination_ailake.writer import StreamWriter

import ailake

BASE        = pathlib.Path(os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')).parent
AIRBYTE_BASE = str(BASE / 'ailake_airbyte_demo')
DIM         = 32

print(f'Base path : {AIRBYTE_BASE}')
print(f'ailake    : {ailake.__version__ if hasattr(ailake, "__version__") else "loaded"}')

## 1. Architecture

```
Airbyte Source  ──▶  AilakeDestination.write()
                          │
                          ├─ for each stream:
                          │     StreamWriter(stream_name, cfg, embedder)
                          │         ├─ extract text_field from record
                          │         ├─ batch records until batch_size
                          │         ├─ embedder.embed(texts) → float32[]
                          │         └─ ailake.open_table(path).insert(texts, embs)
                          │
                          └─ on STATE message:
                                StreamWriter.commit() → Iceberg snapshot
                                yield STATE back to Airbyte
```

Each stream maps to one AI-Lake table at `{table_base_path}/{stream_name}/`.
The table is valid Apache Iceberg Spec v2 — readable by Spark, Trino, DuckDB, PyIceberg
without the AI-Lake SDK.

## 2. Config — `AilakeDestinationConfig`

All Airbyte connector config fields map to `AilakeDestinationConfig` fields.
Validated before any I/O starts.

In [ ]:
cfg = AilakeDestinationConfig.from_dict({
    'table_base_path'         : AIRBYTE_BASE,
    'embed_mode'              : 'cmd',
    'embed_cmd'               : f'{sys.executable} /opt/demo/embed_cmd.py {DIM}',
    'embedding_dim'           : DIM,
    'embedding_metric'        : 'cosine',
    'embedding_model'         : 'demo-embed-v1',
    'embedding_model_version' : '1.0',
    'text_field'              : 'content',
    'batch_size'              : 4,
})

errors = cfg.validate()
print('Validation errors:', errors if errors else 'none — OK')
print()
print(f'table_base_path  : {cfg.table_base_path}')
print(f'embed_mode       : {cfg.embed_mode}')
print(f'embedding_model  : {cfg.embedding_model}@{cfg.embedding_model_version}')
print(f'batch_size       : {cfg.batch_size}')
print(f'table_path(docs) : {cfg.table_path("docs")}')

## 3. CmdEmbedder — local embedding via shell command

`embed_cmd` receives a JSON string array on stdin and must write a JSON float-array-of-arrays
to stdout. The demo uses `embed_cmd.py` — deterministic unit vectors seeded from text hash.

Replace with any real model:
```bash
# OpenAI CLI wrapper
embed_cmd = "python openai_embed.py"
# Or use embed_mode=openai / embed_mode=cohere directly
```

In [ ]:
embedder = build_embedder(cfg)

test_texts = [
    'Apache Iceberg provides ACID transactions for data lakes.',
    'HNSW graphs enable sub-linear approximate nearest neighbour search.',
]
vecs = embedder.embed(test_texts)

print(f'Input texts : {len(test_texts)}')
print(f'Output shape: {vecs.shape}  dtype={vecs.dtype}')
print(f'L2 norm[0]  : {np.linalg.norm(vecs[0]):.4f}  (unit vector)')  

## 4. StreamWriter — batch write + auto-flush

`StreamWriter` buffers records and flushes automatically when `batch_size` is reached.
Each flush: extract `text_field` → embed → `ailake.open_table(path).insert(texts, embs)`.
`commit()` flushes remaining buffer and commits an Iceberg snapshot.

In [ ]:
RECORDS = [
    {'id': i, 'content': text, 'category': cat}
    for i, (text, cat) in enumerate([
        ('Apache Iceberg provides ACID transactions for data lakes.',           'databases'),
        ('HNSW graphs enable sub-linear approximate nearest-neighbour search.', 'algorithms'),
        ('Parquet columnar storage reduces I/O for analytical queries.',         'storage'),
        ('Retrieval-augmented generation grounds LLMs in external knowledge.',  'llm'),
        ('Product quantization compresses high-dimensional vectors by 32-256x.','compression'),
        ('Vector databases power semantic search and recommendation systems.',   'databases'),
        ('Rust zero-cost abstractions enable safe systems programming.',         'languages'),
        ('MinIO provides S3-compatible object storage for on-premise clusters.', 'infrastructure'),
        ('Embedding models convert text to dense vectors for semantic matching.','ml'),
        ('DuckDB runs analytical SQL directly on Parquet files without a server.','databases'),
        ('Apache Arrow in-memory columnar format enables zero-copy IPC.',        'storage'),
        ('Cosine similarity measures the angle between two vectors.',            'algorithms'),
    ])
]

writer = StreamWriter('tech_docs', cfg, embedder)

flush_count_before = 0
for i, rec in enumerate(RECORDS):
    writer.add(rec)
    # batch_size=4 → flush at records 3, 7, 11
    if (i + 1) % cfg.batch_size == 0:
        print(f'  auto-flush after record {i+1}')

snap_id = writer.commit()
print(f'\nCommitted snapshot_id={snap_id}  path={cfg.table_path("tech_docs")}')

## 5. Simulate `Destination.write()` — full connector flow

The Airbyte platform calls `Destination.write(config, catalog, messages)` where
`messages` is a stream of `AirbyteMessage` objects.

On each `STATE` message: all pending writers flush + commit, then STATE is yielded back
to signal Airbyte that data up to this point is durable.

Below we simulate the message stream manually.

In [ ]:
from airbyte_cdk.models import (
    AirbyteMessage, AirbyteRecordMessage, AirbyteStateMessage,
    AirbyteStateType, Type,
)
from airbyte_destination_ailake.destination import AilakeDestination
from airbyte_cdk.models import (
    ConfiguredAirbyteCatalog, ConfiguredAirbyteStream, AirbyteStream as AirbyteStreamModel,
    SyncMode, DestinationSyncMode,
)
import json as _json

import shutil as _shutil
STREAM2_PATH = str(BASE / 'ailake_airbyte_demo2')
_shutil.rmtree(STREAM2_PATH, ignore_errors=True)
cfg2 = AilakeDestinationConfig.from_dict({
    'table_base_path'         : STREAM2_PATH,
    'embed_mode'              : 'cmd',
    'embed_cmd'               : f'{sys.executable} /opt/demo/embed_cmd.py {DIM}',
    'embedding_dim'           : DIM,
    'embedding_model'         : 'demo-embed-v1',
    'text_field'              : 'content',
    'batch_size'              : 5,
})

# airbyte-cdk 7+ uses dataclasses (not Pydantic) — build manually
catalog = ConfiguredAirbyteCatalog(
    streams=[
        ConfiguredAirbyteStream(
            stream=AirbyteStreamModel(
                name='articles',
                json_schema={},
                supported_sync_modes=[SyncMode.full_refresh],
            ),
            sync_mode=SyncMode.full_refresh,
            destination_sync_mode=DestinationSyncMode.overwrite,
        )
    ]
)

def make_record(i: int, text: str) -> AirbyteMessage:
    return AirbyteMessage(
        type=Type.RECORD,
        record=AirbyteRecordMessage(stream='articles', data={'id': i, 'content': text}, emitted_at=0),
    )

def make_state(value: dict) -> AirbyteMessage:
    return AirbyteMessage(
        type=Type.STATE,
        state=AirbyteStateMessage(type=AirbyteStateType.GLOBAL, global_=None, data=value),
    )

messages = [
    make_record(0, 'AI-Lake combines Parquet, HNSW, and Iceberg in one file.'),
    make_record(1, 'Vector search enables semantic retrieval beyond keyword matching.'),
    make_record(2, 'Apache Iceberg time-travel lets you query historical snapshots.'),
    make_state({'offset': 3}),   # <-- triggers flush + commit
    make_record(3, 'DuckDB reads AI-Lake Parquet files with no plugin needed.'),
    make_record(4, 'Cosine distance captures semantic angle between embeddings.'),
]

destination = AilakeDestination()
state_msgs_received = []

for out_msg in destination.write(cfg2.from_dict(cfg2.__dict__ | {'table_base_path': STREAM2_PATH}).__dict__,
                                   catalog, messages):
    if out_msg.type == Type.STATE:
        state_msgs_received.append(out_msg.state.data)
        print(f'STATE emitted back: {out_msg.state.data}  (data durable up to this point)')

print(f'\nTotal STATE messages received: {len(state_msgs_received)}')
print(f'articles table: {STREAM2_PATH}/articles/')

## 6. Search results — AI-Lake vector search on Airbyte-written data

Tables written by the connector are fully searchable with `ailake.search()`.

In [ ]:
table_path = cfg.table_path('tech_docs')

queries = [
    'semantic similarity for information retrieval',
    'ACID transactions and data consistency',
    'storage compression for large embeddings',
]

for query_text in queries:
    q_vec = embedder.embed([query_text])[0].tolist()
    results = ailake.search(table_path, q_vec, top_k=3, fetch_data=True).to_pandas()
    print(f'Query: "{query_text}"')
    for _, row in results.iterrows():
        print(f'  [{row["_distance"]:.4f}] {row["text"][:80]}')
    print()

## 7. DuckDB / Iceberg compat — no AI-Lake SDK needed

Tables written by the connector are valid Apache Iceberg Spec v2.
Any Iceberg-compatible engine reads tabular data normally.
The HNSW footer section is invisible to standard Parquet readers.

In [ ]:
parquet_glob = str(pathlib.Path(table_path) / '**' / '*.parquet')
con = duckdb.connect()

# Count rows and show schema — embedding is BLOB (F16 bytes), all other cols normal
schema_df = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{parquet_glob}', hive_partitioning=false) LIMIT 1"
).df()
print('Schema:')
print(schema_df[['column_name', 'column_type']].to_string(index=False))
print()

In [ ]:
# Full-text filter — predicate pushed down to Parquet
# Note: StreamWriter stores only the text_field (content → text column).
# Extra record fields (id, category) are not persisted by the Airbyte destination.
result = con.execute(
    f"SELECT text FROM read_parquet('{parquet_glob}', hive_partitioning=false)"
    f" WHERE text LIKE '%Parquet%' OR text LIKE '%DuckDB%'"
).df()
print("Records mentioning Parquet or DuckDB:")
result

In [ ]:
# Row count and average embedding byte size
con.execute(
    f"SELECT count(*) AS total_rows, avg(octet_length(embedding)) AS avg_emb_bytes"
    f" FROM read_parquet('{parquet_glob}', hive_partitioning=false)"
).df()

## 8. Embedding model tracking

`embedding_model` and `embedding_model_version` in the connector config flow through to
`ailake.embedding-model` in Iceberg table properties — enabling:

- **Audit**: which model produced each vector, stored per-file in Avro manifest `key_metadata`
- **Migration detection**: `migrate_embeddings()` reads per-file model to find stale files
- **Error messages**: `ModelMismatch` names the stored model when query dim doesn't match

In [ ]:
# Read Iceberg metadata.json — ailake.embedding-model is stored there
meta_dir  = pathlib.Path(table_path) / 'default' / 'table' / 'metadata'
hint      = (meta_dir / 'version-hint.text').read_text().strip()
meta      = json.loads((meta_dir / f'v{hint}.metadata.json').read_text())
props     = meta.get('properties', {})

print('Iceberg table properties — AI-Lake embedding tracking:')
for k in sorted(props):
    if k.startswith('ailake.'):
        print(f'  {k:45s} = {props[k]}')

In [ ]:
# ModelMismatch error — wrong query dimension triggers it
wrong_query = np.random.default_rng(42).standard_normal(64).astype(np.float32)
try:
    ailake.search(table_path, wrong_query.tolist(), top_k=3).to_list()
except Exception as e:
    print(f'ModelMismatch: {e}')
    print('(Connector config embedding_model/embedding_model_version appear in error message)')

In [ ]:
# Simulate model upgrade: re-run connector with new model, then migrate
# In practice: update embedding_model / embedding_model_version in Airbyte config
#              run a new sync → new snapshot with new model
#              or call migrate_embeddings() directly to re-embed in-place

def embed_v2(texts: list[str]) -> list[list[float]]:
    """Simulated v2 model — same dim, different weights."""
    out = []
    for t in texts:
        rng = random.Random((hash(t) ^ 0xDEADBEEF) & 0xFFFFFFFF)
        v   = [rng.gauss(0, 1) for _ in range(DIM)]
        nm  = math.sqrt(sum(x * x for x in v)) or 1.0
        out.append([x / nm for x in v])
    return out

MIGRATE_PATH = str(BASE / 'ailake_airbyte_migrate')

# Write with v1
cfg_v1 = AilakeDestinationConfig.from_dict({
    'table_base_path'         : MIGRATE_PATH,
    'embed_mode'              : 'cmd',
    'embed_cmd'               : f'{sys.executable} /opt/demo/embed_cmd.py {DIM}',
    'embedding_dim'           : DIM,
    'embedding_model'         : 'demo-embed-v1',
    'embedding_model_version' : '1.0',
    'text_field'              : 'content',
    'batch_size'              : 6,
})
w = StreamWriter('articles', cfg_v1, build_embedder(cfg_v1))
for rec in RECORDS[:6]:
    w.add(rec)
w.commit()
print('v1 data written')

# Migrate to v2 in-place
ailake.migrate_embeddings(
    path              = cfg_v1.table_path('articles'),
    old_column        = 'embedding',
    new_column        = 'embedding',
    embed_fn          = embed_v2,
    text_column       = 'text',
    strategy          = 'atomic_replace',
    new_model         = 'demo-embed-v2',
    new_model_version = '2.0',
)
print('Migration to v2 complete')

# Verify updated property
mp  = pathlib.Path(cfg_v1.table_path('articles')) / 'default' / 'table' / 'metadata'
h   = (mp / 'version-hint.text').read_text().strip()
m   = json.loads((mp / f'v{h}.metadata.json').read_text())
print('Post-migration ailake.embedding-model:', m.get('properties', {}).get('ailake.embedding-model'))

## 9. Airbyte platform usage

### Self-hosted Airbyte UI

```bash
# Build and tag the connector image
cd airbyte-destination-ailake
docker build -t airbyte-destination-ailake:0.0.1 .

# Then in Airbyte UI:
# Settings → Destinations → New custom connector
# Name: AI-Lake
# Docker image: airbyte-destination-ailake:0.0.1
# The JSON spec auto-generates the config form
```

### PyAirbyte (local, with Docker)

```python
import airbyte as ab

source = ab.get_source(
    "source-postgres",
    config={"host": "...", "database": "...", "username": "...", "password": "..."},
)

destination = ab.get_destination(
    "destination-ailake",
    config={
        "table_base_path": "s3://my-lake/airbyte",
        "embed_mode": "openai",
        "openai_api_key": "sk-...",
        "embedding_model": "text-embedding-3-small",
        "embedding_dim": 1536,
    },
    docker_image="airbyte-destination-ailake:0.0.1",
)

# Syncs Postgres → AI-Lake (Docker required)
ab.move_data(source=source, destination=destination)
```

### Without Docker — use classes directly (this notebook's approach)

```python
from airbyte_destination_ailake.config import AilakeDestinationConfig
from airbyte_destination_ailake.embedder import OpenAIEmbedder
from airbyte_destination_ailake.writer import StreamWriter

cfg      = AilakeDestinationConfig.from_dict({...})
embedder = OpenAIEmbedder(api_key="sk-...", model="text-embedding-3-small")
writer   = StreamWriter("my_stream", cfg, embedder)

for record in my_records:
    writer.add(record)
writer.commit()
```

## Next Steps

| Resource | Location |
|---|---|
| Connector package | `airbyte-destination-ailake/` |
| Config reference | `airbyte-destination-ailake/airbyte_destination_ailake/spec.json` |
| Standalone scripts | `airbyte-destination-ailake/demo/demo_local.py` / `demo_openai.py` |
| Unit tests | `airbyte-destination-ailake/tests/test_destination.py` |
| AI-Lake SDK overview | `01_ailake_demo.ipynb` |
| DuckDB compat | `02_duckdb.ipynb` |
| Apache Spark | `03_spark.ipynb` |
| Trino | `04_trino.ipynb` |